In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

# Novas colunas

In [ ]:
df_clean = pd.read_csv('../processed/df_data_clean.csv')
df_clean

In [ ]:
df_clean.columns

In [ ]:
df_novascolunas = df_clean.copy()
#-------------------------------------------------------------------------
df_novascolunas['sqft_lot_more_living'] = (df_novascolunas['sqft_living'] + df_novascolunas['sqft_lot']).round(2)
df_novascolunas['living_percent_total'] = (df_novascolunas['sqft_living'] / df_novascolunas['sqft_lot_more_living']).round(2)
#-------------------------------------------------------------------------
df_novascolunas['percent_construcao_above'] = (df_novascolunas['sqft_above']/df_novascolunas['sqft_living']).round(2)
#-------------------------------------------------------------------------
df_novascolunas['lot_15_compare'] =  (df_novascolunas['sqft_lot']/df_novascolunas['sqft_lot15']).round(2)
df_novascolunas['living_15_compare'] = (df_novascolunas['sqft_living']/df_novascolunas['sqft_living15']).round(2)
#-------------------------------------------------------------------------
df_novascolunas['mean_price_zipcode'] = (df_novascolunas.groupby('zipcode')['price'].transform('mean')).round(2)
#-------------------------------------------------------------------------
df_novascolunas['age'] = 2026 - df_novascolunas['yr_built']

df_novascolunas['age_renovated'] = np.where(
    df_novascolunas['yr_renovated'] == 0, 
    df_novascolunas['age'], 
    2026 - df_novascolunas['yr_renovated']
).round(2)
#-------------------------------------------------------------------------
df_novascolunas['bath_per_bed'] = np.where(
    df_novascolunas['bedrooms'] > 0,
    (df_novascolunas['bathrooms'] / df_novascolunas['bedrooms']).round(2),
    df_novascolunas['bathrooms'])
df_novascolunas['bed_per_floor'] = np.where(
    df_novascolunas['bedrooms'] > 0,
    (df_novascolunas['bedrooms'] / df_novascolunas['floors']).round(2),
    df_novascolunas['bedrooms'])
#-------------------------------------------------------------------------
df_novascolunas['grade_x_condition'] = df_novascolunas['grade'] * df_novascolunas['condition']
#-------------------------------------------------------------------------
df_novascolunas['bool_renovated'] = np.where(df_novascolunas['yr_renovated'] > 0, 1, 0)
df_novascolunas['bool_basement'] = np.where(df_novascolunas['sqft_basement'] > 0, 1, 0)
df_novascolunas['living_more_than_lot'] = np.where(df_novascolunas['sqft_living'] >= df_novascolunas['sqft_lot'], 1, 0)
#-------------------------------------------------------------------------
seattle_lat, seattle_long = 47.6062, -122.3321
df_novascolunas['dist_seattle'] = np.sqrt(
    (df_novascolunas['lat'] - seattle_lat)**2 + 
    (df_novascolunas['long'] - seattle_long)**2
).round(4)


In [ ]:
# Confirmando range
df_novascolunas['age_renovated'].describe()

In [ ]:
df_novascolunas['bool_renovated_45'] = np.where(df_novascolunas['age_renovated'] < 45, 1, 0)
df_novascolunas['bool_renovated_15'] = np.where(df_novascolunas['age_renovated'] < 15, 1, 0)

In [ ]:
df_novascolunas.to_csv('../processed/df_novascolunas.csv', index=False, sep=';', encoding='utf-8-sig')

# Analise Exploratória

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(df_colunas_necessarias.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação - O GPS do seu Dataset')
plt.tight_layout()
plt.show()

In [ ]:
df_eda = df_sem_vazamento.copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico da esquerda: Preço original
sns.histplot(df_eda['price'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribuição Original do Preço (Altamente Assimétrica)')
axes[0].set_xlabel('Preço ($)')

# Gráfico da direita: Preço escalado em Logaritmo
sns.histplot(np.log1p(df_eda['price']), kde=True, ax=axes[1], color='olive')
axes[1].set_title('Distribuição do Preço Normalizada (Com Log)')
axes[1].set_xlabel('Log do Preço')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_eda, x='sqft_living', y='price', alpha=0.5, color='purple')

plt.title('Relação Linear: Área Interna (sqft_living) vs Preço')
plt.xlabel('Área Interna (sqft_living)')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_eda, x='grade', y='price', alpha=0.5, color='blue')

plt.title('Relação Linear: Área Interna (sqft_living) vs Preço')
plt.xlabel('Grade')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_eda, x='sqft_above', y='price', alpha=0.5, color='coral')

plt.title('Relação Linear: Área Interna (sqft_living) vs Preço')
plt.xlabel('Contrução acima do solo')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_eda, x='sqft_living15', y='price', alpha=0.5, color='red')

plt.title('Relação Linear: Área Interna (sqft_living) vs Preço')
plt.xlabel('Contrução média das casas próximas')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_eda, x='mean_price_zipcode', y='price', alpha=0.5, color='green')

plt.title('Relação Linear: Área Interna (sqft_living) vs Preço')
plt.xlabel('Preço médio por zipcode')
plt.ylabel('Preço ($)')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Filtra apenas as colunas numéricas (ignora IDs se houver)
colunas_numericas = df_eda.select_dtypes(include=[np.number]).columns.tolist()
if 'id' in colunas_numericas:
    colunas_numericas.remove('id')

# Dicionário para guardar as tabelas com as linhas inteiras de outliers de cada coluna
outliers_por_coluna = {}

print("=== INICIANDO ANÁLISE DE OUTLIERS POR COLUNA ===\n")

for col in colunas_numericas:
    # Ignora colunas binárias ou categóricas com poucos valores únicos (ex: waterfront, floors)
    if df_eda[col].nunique() <= 5:
        continue
        
    # Remove nulos apenas para o cálculo estatístico
    dados_limpos = df_eda[col].dropna()
    
    # 3. Teste de Normalidade (D'Agostino & Pearson)
    # H0: Os dados seguem uma distribuição normal (se p_valor > 0.05, aceitamos H0)
    stat, p_valor = stats.normaltest(dados_limpos)
    alpha = 0.05
    dist_normal = p_valor > alpha
    
    # 4. Define o método de detecção baseado na normalidade
    if dist_normal:
        # Distribuição Normal -> Z-Score (Moderação de 3 desvios padrão)
        z_scores = np.abs(stats.zscore(dados_limpos))
        mascara_outliers = z_scores > 3
        metodo = "Z-Score (Normal)"
    else:
        # Distribuição Não-Normal -> IQR (Amplitude Interquartílica)
        q1 = dados_limpos.quantile(0.25)
        q3 = dados_limpos.quantile(0.75)
        iqr = q3 - q1
        limite_inferior = q1 - 1.5 * iqr
        limite_superior = q3 + 1.5 * iqr
        mascara_outliers = (dados_limpos < limite_inferior) | (dados_limpos > limite_superior)
        metodo = "IQR (Não-Normal)"
        
    # 5. Extrai as LINHAS INTEIRAS do DataFrame original que são outliers nesta coluna
    linhas_outliers = df_eda[df_eda[col].isin(dados_limpos[mascara_outliers])]
    
    # Salva no dicionário para você poder acessar depois
    outliers_por_coluna[col] = linhas_outliers
    
    # Exibe o diagnóstico no console
    print(f"Coluna: '{col}'")
    print(f"  - Distribuição Normal? {'SIM' if dist_normal else 'NÃO'} (p-valor: {p_valor:.5f})")
    print(f"  - Método aplicado: {metodo}")
    print(f"  - Total de outliers nesta coluna: {len(linhas_outliers)} linhas")
    print("-" * 60)

print("\n=== ANÁLISE CONCLUÍDA ===")
print("Para acessar as linhas inteiras de outliers de uma coluna específica, use:")
print("outliers_por_coluna['nome_da_coluna']")

# Limpeza de colunas

In [ ]:
df_novascolunas.columns

In [ ]:
limpeza_colunas = ['id', 'date','yr_built','zipcode','lat','long','price_for_living','price_for_lot','living_price_for_basement','living_price_for_above','zipcode_compare' ]

df_colunas_necessarias = df_novascolunas.drop(columns=limpeza_colunas)
df_colunas_necessarias

In [ ]:
df_colunas_necessarias.to_csv('../processed/df_sem_vazamento.csv', index=False, sep=';', encoding='utf-8-sig')